# Notebook Overview — Prepare Video Data

## Purpose

This notebook prepares public VideoQA benchmark datasets for iterative RAG-based Video Question Answering (VideoQA) experimentation. The workflow downloads, organizes, verifies, and configures video datasets, metadata resources, captions, and question-answer annotations required for downstream preprocessing, embedding generation, retrieval, and inference workflows.

## Inputs

* Public VideoQA benchmark datasets (e.g., MSVD-QA, TGIF-QA)
* Video files
* Question-answer annotation files
* Caption and metadata resources
* User configuration settings
* Project configuration modules

## Outputs

* Organized dataset directory structure
* Downloaded and verified video assets
* Prepared metadata and annotation files
* Dataset configuration information
* Runtime environment verification results

## Processing Workflow

* Configure runtime environment and project settings
* Download or verify required benchmark datasets
* Organize dataset directories and metadata resources
* Validate dataset structure and required files
* Display dataset statistics and verification summaries

## Notes

* This notebook focuses on dataset preparation and validation only.
* Frame extraction, embedding generation, vector indexing, and retrieval workflows are performed in later notebooks.
* Public dataset downloads may require significant storage space and execution time depending on dataset selection.


### 🔷 Step 1 — Clone Required Repository Files

* Clone only the repository files required for notebook execution using sparse checkout.
* Authenticate access to the private repository using a GitHub fine-grained token stored in Google Colab Secrets.
* Configure the local runtime workspace and change to the repository working directory.
* Verify that required configuration files are available before continuing notebook execution.
* Optionally display repository paths and cloned files when `VERBOSE=True`.


In [1]:
# ============================================================
# Step 1: Clone Required Repository Files
# ============================================================

VERBOSE = True    # True or False as desired

import os
from google.colab import userdata

REPO_NAME = "iterative-video-rag"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# ------------------------------------------------------------
# Retrieve GitHub Token from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

# ------------------------------------------------------------
# Move to Base Directory
# ------------------------------------------------------------

%cd {REPO_BASE_DIR}

# ------------------------------------------------------------
# Clone Repository if Needed
# ------------------------------------------------------------

if not os.path.exists(REPO_DIR):

    if VERBOSE:
        print("Cloning required repository files...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    %cd {REPO_DIR}

    !git sparse-checkout init --no-cone
    !git sparse-checkout set src/iterative_rag_config.py
    !git checkout --quiet main

else:

    if VERBOSE:
        print(f"Repository already exists: {REPO_DIR}")

    %cd {REPO_DIR}

# ------------------------------------------------------------
# Verify Repository Setup
# ------------------------------------------------------------

if not os.path.exists("src/iterative_rag_config.py"):
    raise FileNotFoundError(
        "Required file not found: src/iterative_rag_config.py"
    )

print("Repository setup complete.")

if VERBOSE:
    print(f"Current directory: {os.getcwd()}")
    print("\nAvailable files:")
    !find src -maxdepth 2 -type f | sort



/content
Cloning required repository files...
/content/iterative-video-rag
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 456 bytes | 456.00 KiB/s, done.
Repository setup complete.
Current directory: /content/iterative-video-rag

Available files:
src/iterative_rag_config.py


### 🔷 Step 2 — Import Project Configuration

* Import centralized project configuration settings and constants.
* Load dataset paths, directory definitions, and runtime parameters.
* Initialize reusable configuration values shared across notebooks.
* Verify that required configuration files are accessible.

In [2]:
# ============================================
# Step 2: Import Project Configuration
# ============================================

# -------------------------------------------------
# Import centralized project configuration values
# -------------------------------------------------
try:
    from src.iterative_rag_config import *

except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "Project configuration could not be imported. "
        "Verify that the repository was cloned correctly and that "
        "'src/project_config.py' exists."
    ) from error


# -------------------------------------------------
# Verify that key configuration values are available
# -------------------------------------------------
required_config_values = [
    "BASE_DIR",
    "DATA_DIR",
    "DATASETS_DIR",
    "METADATA_DIR",
]

missing_config_values = [
    name for name in required_config_values
    if name not in globals()
]

if missing_config_values:
    raise ValueError(
        "Missing required configuration values: "
        + ", ".join(missing_config_values)
    )


# -------------------------------------------------
# Display configuration summary
# -------------------------------------------------
print("Project configuration imported successfully.")

if VERBOSE:
    print(f"BASE_DIR:      {BASE_DIR}")
    print(f"DATA_DIR:      {DATA_DIR}")
    print(f"DATASETS_DIR:  {DATASETS_DIR}")
    print(f"METADATA_DIR:  {METADATA_DIR}")



Project configuration imported successfully.
BASE_DIR:      /content/iterative-video-rag
DATA_DIR:      /content/iterative-video-rag/data
DATASETS_DIR:  /content/iterative-video-rag/data/datasets
METADATA_DIR:  /content/iterative-video-rag/data/metadata


### 🔷 Step 3 — Configure Runtime Environment

* Configure the notebook runtime environment and workspace settings.
* Initialize required directories, environment variables, and project paths.
* Verify that required project directories exist and are accessible.
* Display runtime environment and directory verification summaries.
* Prepare the notebook environment for dataset preparation workflows.


In [3]:
# ============================================
# Step 3: Configure Runtime Environment
# ============================================

# -------------------------------------------------
# Import runtime support modules
# -------------------------------------------------
import os
import platform
from pathlib import Path


# -------------------------------------------------
# Initialize required project directories
# -------------------------------------------------
required_directories = [
    DATA_DIR,
    DATASETS_DIR,
    METADATA_DIR,
]

for directory_path in required_directories:
    Path(directory_path).mkdir(parents=True, exist_ok=True)


# -------------------------------------------------
# Configure basic environment variables
# -------------------------------------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# -------------------------------------------------
# Display runtime environment summary
# -------------------------------------------------
print("Runtime environment configured successfully.")

if VERBOSE:
    print(f"Python version: {platform.python_version()}")
    print(f"Platform:       {platform.platform()}")
    print()
    print("Verified directories:")

    for directory_path in required_directories:
        print(f"  - {directory_path}")



Runtime environment configured successfully.
Python version: 3.12.13
Platform:       Linux-6.6.122+-x86_64-with-glibc2.35

Verified directories:
  - /content/iterative-video-rag/data
  - /content/iterative-video-rag/data/datasets
  - /content/iterative-video-rag/data/metadata


### 🔷 Step 4 — Configure Dataset Selection

* Select the benchmark datasets to verify and prepare.
* Define enabled datasets and associated storage locations.
* Load dataset-specific metadata and directory configuration settings.
* Verify selected dataset configuration settings before execution.


In [6]:
# ============================================
# Step 4: Configure Dataset Selection
# ============================================

# -------------------------------------------------
# Select benchmark datasets for preparation
# -------------------------------------------------
ENABLED_DATASETS = [
    "MSVD-QA",
    "TGIF-QA",
]

# -------------------------------------------------
# Verify selected dataset configuration
# -------------------------------------------------
missing_dataset_configs = [
    dataset_name for dataset_name in ENABLED_DATASETS
    if dataset_name not in DATASET_CONFIG
]

if missing_dataset_configs:
    raise ValueError(
        "Missing dataset configuration for: "
        + ", ".join(missing_dataset_configs)
    )


# -------------------------------------------------
# Display dataset selection summary
# -------------------------------------------------
print("Dataset selection configured successfully.")

if VERBOSE:
    print(f"Enabled datasets: {', '.join(ENABLED_DATASETS)}")



Dataset selection configured successfully.
Enabled datasets: MSVD-QA, TGIF-QA


### 🔷 Step 5 — Download or Verify Benchmark Datasets

* Download required benchmark datasets when local copies are unavailable.
* Verify expected dataset directory locations and storage paths.
* Detect missing datasets or incomplete dataset preparation steps.
* Display dataset download, availability, and verification summaries.
* Prepare the project workspace for subsequent dataset organization and validation workflows.

In [11]:
# ============================================
# Step 5: Download or Verify Benchmark Datasets
# ============================================

# -------------------------------------------------
# Import dataset preparation support modules
# -------------------------------------------------
from pathlib import Path


# -------------------------------------------------
# Verify dataset availability
# -------------------------------------------------
dataset_status = {}

for dataset_name in ENABLED_DATASETS:

    dataset_dir = DATASET_CONFIG[dataset_name]["dataset_dir"]

    dataset_exists = dataset_dir.exists()

    dataset_status[dataset_name] = {
        "dataset_dir": dataset_dir,
        "exists": dataset_exists,
    }

    # -------------------------------------------------
    # Report missing datasets
    # -------------------------------------------------
    if not dataset_exists:

        print(f"[WARNING] Dataset not found: {dataset_name}")
        print(f"          Expected path: {dataset_dir}")

        # -------------------------------------------------
        # Placeholder for future download support
        # -------------------------------------------------
        print(f"          Automatic download not yet implemented.")


# -------------------------------------------------
# Display dataset verification summary
# -------------------------------------------------
print("Dataset verification completed.")

if VERBOSE:

    for dataset_name, status in dataset_status.items():

        print()
        print(f"Dataset: {dataset_name}")
        print(f"Path:    {status['dataset_dir']}")
        print(f"Exists:  {status['exists']}")



Dataset verification completed.

Dataset: MSVD-QA
Path:    /content/iterative-video-rag/data/datasets/MSVD-QA
Exists:  True

Dataset: TGIF-QA
Path:    /content/iterative-video-rag/data/datasets/TGIF-QA
Exists:  True


### 🔷 Step 6 — Organize Dataset Directories and Metadata

* Create and organize dataset directory structures for videos, questions, and metadata.
* Initialize standardized paths used throughout the VideoQA pipeline.
* Prepare metadata resources required for dataset validation and preprocessing.
* Verify that expected dataset folders and metadata files exist.


In [9]:
# ============================================
# Step 6: Organize Dataset Directories and Metadata
# ============================================

# -------------------------------------------------
# Create expected dataset subdirectories
# -------------------------------------------------
for dataset_name in ENABLED_DATASETS:

    dataset_paths = DATASET_CONFIG[dataset_name]

    required_dataset_directories = [
        dataset_paths["dataset_dir"],
        dataset_paths["videos_dir"],
        dataset_paths["questions_dir"],
        dataset_paths["metadata_dir"],
    ]

    for directory_path in required_dataset_directories:
        directory_path.mkdir(parents=True, exist_ok=True)


# -------------------------------------------------
# Verify organized dataset directories
# -------------------------------------------------
dataset_directory_status = {}

for dataset_name in ENABLED_DATASETS:

    dataset_paths = DATASET_CONFIG[dataset_name]

    dataset_directory_status[dataset_name] = {
        "dataset_dir": dataset_paths["dataset_dir"].exists(),
        "videos_dir": dataset_paths["videos_dir"].exists(),
        "questions_dir": dataset_paths["questions_dir"].exists(),
        "metadata_dir": dataset_paths["metadata_dir"].exists(),
    }


# -------------------------------------------------
# Display directory organization summary
# -------------------------------------------------
print("Dataset directories organized successfully.")

if VERBOSE:

    for dataset_name, status in dataset_directory_status.items():

        print()
        print(f"Dataset: {dataset_name}")

        for directory_name, exists in status.items():
            print(f"{directory_name}: {exists}")



Dataset directories organized successfully.

Dataset: MSVD-QA
dataset_dir: True
videos_dir: True
questions_dir: True
metadata_dir: True

Dataset: TGIF-QA
dataset_dir: True
videos_dir: True
questions_dir: True
metadata_dir: True


### 🔷 Step 7 — Validate Dataset Structure and Required Files

* Validate dataset directory structures and required resource files.
* Verify the presence of videos, question files, annotations, and metadata resources.
* Detect missing files, invalid paths, or incomplete dataset preparation steps.
* Generate validation summaries for each selected benchmark dataset.


In [10]:
# ============================================
# Step 7: Validate Dataset Structure and Required Files
# ============================================

# -------------------------------------------------
# Validate dataset contents
# -------------------------------------------------
dataset_validation_status = {}

for dataset_name in ENABLED_DATASETS:

    dataset_paths = DATASET_CONFIG[dataset_name]

    videos_dir = dataset_paths["videos_dir"]
    questions_dir = dataset_paths["questions_dir"]
    metadata_dir = dataset_paths["metadata_dir"]

    video_files = list(videos_dir.glob("*")) if videos_dir.exists() else []
    question_files = list(questions_dir.glob("*")) if questions_dir.exists() else []
    metadata_files = list(metadata_dir.glob("*")) if metadata_dir.exists() else []

    dataset_validation_status[dataset_name] = {
        "videos_dir_exists": videos_dir.exists(),
        "questions_dir_exists": questions_dir.exists(),
        "metadata_dir_exists": metadata_dir.exists(),
        "video_file_count": len(video_files),
        "question_file_count": len(question_files),
        "metadata_file_count": len(metadata_files),
        "is_ready": (
            videos_dir.exists()
            and questions_dir.exists()
            and metadata_dir.exists()
            and len(video_files) > 0
            and len(question_files) > 0
        ),
    }


# -------------------------------------------------
# Display dataset validation summary
# -------------------------------------------------
print("Dataset structure validation completed.")

if VERBOSE:

    for dataset_name, status in dataset_validation_status.items():

        print()
        print(f"Dataset: {dataset_name}")
        print(f"Videos directory exists:    {status['videos_dir_exists']}")
        print(f"Questions directory exists: {status['questions_dir_exists']}")
        print(f"Metadata directory exists:  {status['metadata_dir_exists']}")
        print(f"Video files found:          {status['video_file_count']}")
        print(f"Question files found:       {status['question_file_count']}")
        print(f"Metadata files found:       {status['metadata_file_count']}")
        print(f"Dataset ready:              {status['is_ready']}")



Dataset structure validation completed.

Dataset: MSVD-QA
Videos directory exists:    True
Questions directory exists: True
Metadata directory exists:  True
Video files found:          0
Question files found:       0
Metadata files found:       0
Dataset ready:              False

Dataset: TGIF-QA
Videos directory exists:    True
Questions directory exists: True
Metadata directory exists:  True
Video files found:          0
Question files found:       0
Metadata files found:       0
Dataset ready:              False


### 🔷 Step 8 — Display Dataset Statistics and Verification Summary

* Display dataset statistics for videos, questions, annotations, and metadata resources.
* Summarize dataset preparation, validation, and verification results.
* Report dataset storage usage, file counts, and directory organization details.
* Highlight missing resources, warnings, or preparation issues when detected.


In [ ]:
# ============================================================
# Step 8: Display Dataset Statistics and Verification Summary
# ============================================================

